> Saved output note: repetitive vLLM/transformers log lines (per-layer bf16->fp16 weight-cast warnings, CUDA-graph and progress bars) were trimmed from the saved outputs. All benchmark results, the T4 UUID line, KV-cache sizing and the own-prompt generation are unchanged.

# HW4.5 Part D — re-run + measured KV cache (Colab T4)
Run on the **same GPU type as the first Part D run** (Runtime > Change runtime type > T4). Run all cells top to bottom (~15 min).
This notebook adds what the first run lacked:
1. **Measured** KV-cache memory vs context length at batch 8 (real GPU allocations, not a formula).
2. A **second, independent run** of the serving benchmark (HF batch 1, vLLM batch 1/8/32) with the same code, compared against the first run.
Outputs: `partD_results_run2.json` (downloads automatically).

In [1]:
!pip -q install vllm
import subprocess; print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)  # UUID capture

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.2.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
GPU 0: Tesla T4 (UUID: GPU-61a63d79-7a2b-8082-e4f3-4e0da13ffe26)


In [2]:
import os, time, json, gc
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
import torch
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MY_PROMPT = "Explain in simple terms why increasing GPU batch size can improve training throughput at first, but eventually gives smaller speed gains and can run out of GPU memory. Include how GPU utilization and memory limits contribute to this behavior."
NEW_TOKENS, REPS = 128, 20
GPU = torch.cuda.get_device_name(0)
print(GPU, round(torch.cuda.get_device_properties(0).total_memory/2**30,1), "GB")
prompts_pool = [MY_PROMPT] + [f"Write a short paragraph about topic number {i}: the history of computing." for i in range(1, 64)]
def pct(xs, p):
    xs = sorted(xs); k = (len(xs)-1)*p/100; f = int(k); c = min(f+1, len(xs)-1)
    return xs[f] + (xs[c]-xs[f])*(k-f)
results = {"run": "HW4.5 re-run (second independent run)", "gpu": GPU, "model": MODEL, "new_tokens": NEW_TOKENS, "reps": REPS}
# First-run numbers (partD_results.json) for the comparison at the end
RUN1 = {"hf_b1": (22.487, 5482.2, 6634.9), "vllm_b1": (65.902, 1893.2, 1981.0),
        "vllm_b8": (501.909, 2041.6, 2045.0), "vllm_b32": (1399.464, 2928.1, 2952.0)}

Tesla T4 14.6 GB


## 1. Measured KV-cache memory vs context length (batch 8)
Method: run the model's prefill on 8 sequences of length *ctx* with `use_cache=True`, keep the returned KV cache alive, and read the GPU memory actually
allocated for it (`torch.cuda.memory_allocated` after minus before). Also record the prefill peak memory and the formula value for comparison.
The first context length that raises an out-of-memory error marks the Hugging Face eager-path limit at batch 8 on this card.

In [3]:
from transformers import AutoModelForCausalLM
m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda().eval()
cfg = m.config
per_tok = 2*cfg.num_hidden_layers*cfg.num_key_value_heads*(getattr(cfg,"head_dim",None) or cfg.hidden_size//cfg.num_attention_heads)*2
BATCH = 8; kv_rows = []
for ctx in (256, 512, 1024, 2048, 4096, 8192, 16384, 32760):
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
    before = torch.cuda.memory_allocated(); torch.cuda.reset_peak_memory_stats()
    row = {"context": ctx, "batch": BATCH, "computed_kv_gb": BATCH*ctx*per_tok/2**30}
    try:
        ids = torch.randint(1000, 5000, (BATCH, ctx), device="cuda")
        with torch.no_grad():
            try: out = m(input_ids=ids, use_cache=True, logits_to_keep=1)
            except TypeError: out = m(input_ids=ids, use_cache=True, num_logits_to_keep=1)
        torch.cuda.synchronize()
        peak = torch.cuda.max_memory_allocated() - before
        logits = out.logits; lbytes = logits.numel()*logits.element_size()
        out_logits_removed = None
        kv = out.past_key_values
        after = torch.cuda.memory_allocated()
        row["measured_kv_gb"] = (after - before - ids.numel()*ids.element_size() - lbytes)/2**30
        row["prefill_peak_gb"] = peak/2**30
        del out, kv, logits, ids
    except torch.cuda.OutOfMemoryError:
        row["measured_kv_gb"] = None; row["prefill_peak_gb"] = None; row["status"] = "OOM"
    kv_rows.append(row); print(row)
    if row.get("status") == "OOM": break
results["kv_measured_b8"] = kv_rows; results["kv_bytes_per_token"] = per_tok
del m; gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

{'context': 256, 'batch': 8, 'computed_kv_gb': 0.0546875, 'measured_kv_gb': 0.0626220703125, 'prefill_peak_gb': 0.1916675567626953}
{'context': 512, 'batch': 8, 'computed_kv_gb': 0.109375, 'measured_kv_gb': 0.109375, 'prefill_peak_gb': 0.47005128860473633}
{'context': 1024, 'batch': 8, 'computed_kv_gb': 0.21875, 'measured_kv_gb': 0.21875, 'prefill_peak_gb': 1.3639302253723145}
{'context': 2048, 'batch': 8, 'computed_kv_gb': 0.4375, 'measured_kv_gb': 0.4375, 'prefill_peak_gb': 4.423172473907471}
{'context': 4096, 'batch': 8, 'computed_kv_gb': 0.875, 'measured_kv_gb': None, 'prefill_peak_gb': None, 'status': 'OOM'}


## 2. Serving re-run (same code as the first run)

In [4]:
from transformers import pipeline, AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
pipe = pipeline("text-generation", model=MODEL, device=0, dtype=torch.float16)
gen = dict(max_new_tokens=NEW_TOKENS, min_new_tokens=NEW_TOKENS, do_sample=False, return_full_text=False)
pipe(prompts_pool[0], **gen)
lat, toks = [], []
for i in range(REPS):
    t = time.time(); out = pipe(prompts_pool[i % len(prompts_pool)], **gen)[0]["generated_text"]; torch.cuda.synchronize()
    lat.append(time.time()-t); toks.append(len(tok(out).input_ids))
results["hf_b1"] = dict(tok_s=sum(toks)/sum(lat), p50_ms=pct(lat,50)*1e3, p95_ms=pct(lat,95)*1e3); print(results["hf_b1"])
del pipe; gc.collect(); torch.cuda.empty_cache()

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'min_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


{'tok_s': 19.89922455573853, 'p50_ms': 5537.7877950668335, 'p95_ms': 10590.423727035522}


In [5]:
from vllm import LLM, SamplingParams
llm = LLM(MODEL, dtype="float16", gpu_memory_utilization=0.85, max_model_len=2048)
sp = SamplingParams(max_tokens=NEW_TOKENS, min_tokens=NEW_TOKENS, ignore_eos=True, temperature=0.0)
llm.generate(prompts_pool[:2], sp)
for B in (1, 8, 32):
    lat, ntok = [], 0
    for r in range(REPS):
        ps = [prompts_pool[(r*B+j) % len(prompts_pool)] for j in range(B)]
        t = time.time(); outs = llm.generate(ps, sp, use_tqdm=False); dt = time.time()-t
        lat.append(dt); ntok += sum(len(o.outputs[0].token_ids) for o in outs)
    results[f"vllm_b{B}"] = dict(tok_s=ntok/sum(lat), p50_ms=pct(lat,50)*1e3, p95_ms=pct(lat,95)*1e3); print(B, results[f"vllm_b{B}"])
cc = getattr(llm.llm_engine, "cache_config", None) or llm.llm_engine.vllm_config.cache_config
results["vllm_kv_capacity_tokens"] = cc.num_gpu_blocks * cc.block_size
print("vLLM KV capacity (tokens):", results["vllm_kv_capacity_tokens"])

(MainProcess pid=586) INFO 10-31 00:16:25 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'gpu_memory_utilization': 0.85, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
(MainProcess pid=586) WARNING 10-31 00:16:25 [arg_utils.py:1945] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
(MainProcess pid=586) INFO 10-31 00:17:00 [model.py:733] Resolved architecture: Qwen2ForCausalLM
(MainProcess pid=586) WARNING 10-31 00:17:00 [model.py:2498] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=586) INFO 10-31 00:17:00 [model.py:2152] Using max model len 2048
(MainProcess pid=586) INFO 10-31 00:17:00 [scheduler.py:307] Chunked prefill is enabled with max_num_batched_tokens=8192.
(MainProcess pid=586) INFO 10-31 00:17:00 [kernel.py:467] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['nat

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

(MainProcess pid=586) INFO 10-31 00:17:04 [core.py:133] Initializing a V1 LLM engine (v0.32.0) with config: model='Qwen/Qwen2.5-1.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-1.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, aux_output_config=AuxOutputConfig(enable_return_routed_experts=False, max_bytes=None), kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityCon

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(MainProcess pid=586) INFO 10-31 00:17:32 [default_loader.py:501] Loading weights took 20.53 seconds
(MainProcess pid=586) INFO 10-31 00:17:34 [model_runner.py:424] Model loading took 2.98 GiB memory and 27.227758 seconds
(MainProcess pid=586) WARNING 10-31 00:17:34 [topk_topp_sampler.py:131] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(MainProcess pid=586) INFO 10-31 00:17:34 [utils.py:347] Using LBNHC KV cache layout.
(MainProcess pid=586) INFO 10-31 00:17:38 [monitor.py:40] [backbone] Starting torch.compile
(MainProcess pid=586) INFO 10-31 00:17:46 [backends.py:1104] [backbone] Using cache directory: /root/.cache/vllm/torch_compile_cache/fdd7859a86/rank_0_0/backbone for vLLM's torch.compile
(MainProcess pid=586) INFO 10-31 00:17:46 [backends.py:1166] [backbone] Dynamo bytecode transform time: 8.11 s


... (3 outputs left out)

(MainProcess pid=586) INFO 10-31 00:18:50 [model_runner.py:1127] Graph capturing finished in 7 secs, took 0.15 GiB


(MainProcess pid=586) INFO 10-31 00:18:50 [gpu_worker.py:971] CUDA graph pool memory: 0.15 GiB (actual), 0.34 GiB (estimated), difference: 0.19 GiB (130.3%).
(MainProcess pid=586) INFO 10-31 00:18:50 [gpu_worker.py:1034] Free memory on device (14.42/14.56 GiB) on startup. Desired GPU memory utilization is (0.85, 12.38 GiB). Actual usage is 3.2 GiB for consumed memory (weights + non-torch), 0.83 GiB for peak activation, and 0.15 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=8652617421` (8.06 GiB) to fit into requested memory, or `--kv-cache-memory=10842073600` (10.1 GiB) to fully utilize gpu memory. Current kv cache memory in use is 8.35 GiB.
(MainProcess pid=586) INFO 10-31 00:18:52 [jit_monitor.py:84] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(MainProcess pid=586) INFO 10-31 00:18:53 [core.py:395] init engine (profile, create kv cache, warmup model) took 78.38 s (compilation: 27.05 s)
(MainPr

Rendering prompts:   0%|          | 0/2 [00:00<?, ?it/s]

(MainProcess pid=586) WARNING 10-31 00:18:56 [jit_monitor.py:140] Triton kernel JIT compilation during inference: kernel_unified_attention. This causes a latency spike; consider extending warmup to cover this shape/config.
1 {'tok_s': 44.03095918944659, 'p50_ms': 2906.770944595337, 'p95_ms': 3145.4111576080327}
8 {'tok_s': 481.20357695291136, 'p50_ms': 2126.5472173690796, 'p95_ms': 2137.978410720825}
32 {'tok_s': 1322.0296615696034, 'p50_ms': 3099.121332168579, 'p95_ms': 3123.3867406845093}
vLLM KV capacity (tokens): 312816


## 3. Comparison with the first run, and save

In [6]:
rows = [("HF pipeline, batch 1","hf_b1"),("vLLM, batch 1","vllm_b1"),("vLLM, batch 8","vllm_b8"),("vLLM, batch 32","vllm_b32")]
print("| Configuration | Run1 tok/s | Run2 tok/s | d% | Run1 p50 | Run2 p50 | d% | Run1 p95 | Run2 p95 | d% |\n|---|---|---|---|---|---|---|---|---|---|")
for n,k in rows:
    a,b,c = RUN1[k]; r = results[k]; d = lambda x,y: f"{(y-x)/x*100:+.1f}%"
    print(f"| {n} | {a:.1f} | {r['tok_s']:.1f} | {d(a,r['tok_s'])} | {b:.0f} | {r['p50_ms']:.0f} | {d(b,r['p50_ms'])} | {c:.0f} | {r['p95_ms']:.0f} | {d(c,r['p95_ms'])} |")
print("\n| Context | Computed KV GB | Measured KV GB | Prefill peak GB |\n|---|---|---|---|")
for r in results["kv_measured_b8"]:
    print(f"| {r['context']} | {r['computed_kv_gb']:.3f} | {('%.3f' % r['measured_kv_gb']) if r['measured_kv_gb'] is not None else 'OOM'} | {('%.2f' % r['prefill_peak_gb']) if r['prefill_peak_gb'] is not None else 'OOM'} |")
json.dump(results, open("partD_results_run2.json","w"), indent=1)
from google.colab import files; files.download("partD_results_run2.json")

| Configuration | Run1 tok/s | Run2 tok/s | d% | Run1 p50 | Run2 p50 | d% | Run1 p95 | Run2 p95 | d% |
|---|---|---|---|---|---|---|---|---|---|
| HF pipeline, batch 1 | 22.5 | 19.9 | -11.5% | 5482 | 5538 | +1.0% | 6635 | 10590 | +59.6% |
| vLLM, batch 1 | 65.9 | 44.0 | -33.2% | 1893 | 2907 | +53.5% | 1981 | 3145 | +58.8% |
| vLLM, batch 8 | 501.9 | 481.2 | -4.1% | 2042 | 2127 | +4.2% | 2045 | 2138 | +4.5% |
| vLLM, batch 32 | 1399.5 | 1322.0 | -5.5% | 2928 | 3099 | +5.8% | 2952 | 3123 | +5.8% |
| Context | Computed KV GB | Measured KV GB | Prefill peak GB |
|---|---|---|---|
| 256 | 0.055 | 0.063 | 0.19 |
| 512 | 0.109 | 0.109 | 0.47 |
| 1024 | 0.219 | 0.219 | 1.36 |
| 2048 | 0.438 | 0.438 | 4.42 |
| 4096 | 0.875 | OOM | OOM |


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import torch, gc, json, subprocess, time
from transformers import AutoModelForCausalLM
print(subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True).stdout)   # UUID capture

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"; BATCH = 8; CHUNK = 128
CHECKPOINTS = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float16).cuda().eval()
cfg = m.config
per_tok = 2*cfg.num_hidden_layers*cfg.num_key_value_heads*(getattr(cfg,"head_dim",None) or cfg.hidden_size//cfg.num_attention_heads)*2
ids = torch.randint(1000, 5000, (BATCH, CHECKPOINTS[-1]), device="cuda")
gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
base = torch.cuda.memory_allocated()          # weights + the id tensor
rows, cache, pos, t0 = [], None, 0, time.time()
try:
    for ctx in CHECKPOINTS:
        torch.cuda.reset_peak_memory_stats()
        while pos < ctx:                       # chunked prefill: never materialises a full 4k+ attention matrix
            with torch.no_grad():
                try: out = m(input_ids=ids[:, pos:pos+CHUNK], past_key_values=cache, use_cache=True, logits_to_keep=1)
                except TypeError: out = m(input_ids=ids[:, pos:pos+CHUNK], past_key_values=cache, use_cache=True, num_logits_to_keep=1)
            cache = out.past_key_values; del out; pos += CHUNK
        torch.cuda.synchronize()
        measured = (torch.cuda.memory_allocated() - base)/2**30
        rows.append({"context": ctx, "batch": BATCH, "computed_kv_gb": BATCH*ctx*per_tok/2**30,
                     "measured_kv_gb": measured, "peak_total_gb": torch.cuda.max_memory_allocated()/2**30})
        print(rows[-1], f"{time.time()-t0:.0f}s elapsed", flush=True)
except torch.cuda.OutOfMemoryError:
    rows.append({"context": ctx, "batch": BATCH, "status": "OOM"}); print("OOM at", ctx)

print("\n| Context | Computed KV GB | Measured KV GB | Peak total GB |\n|---|---|---|---|")
for r in rows:
    print(f"| {r['context']} | " + (f"{r['computed_kv_gb']:.3f} | {r['measured_kv_gb']:.3f} | {r['peak_total_gb']:.2f} |" if "status" not in r else "OOM | OOM | OOM |"))
json.dump({"gpu": torch.cuda.get_device_name(0), "model": MODEL, "batch": BATCH, "chunk": CHUNK,
           "method": "chunked prefill, one pass, KV measured at checkpoints", "rows": rows}, open("kv_chunked_results.json","w"), indent=1)
from google.colab import files; files.download("kv_chunked_results.json")

GPU 0: Tesla T4 (UUID: GPU-61a63d79-7a2b-8082-e4f3-4e0da13ffe26)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

{'context': 256, 'batch': 8, 'computed_kv_gb': 0.0546875, 'measured_kv_gb': 0.0626220703125, 'peak_total_gb': 3.003903388977051} 2s elapsed
{'context': 512, 'batch': 8, 'computed_kv_gb': 0.109375, 'measured_kv_gb': 0.158203125, 'peak_total_gb': 3.100003242492676} 2s elapsed
{'context': 1024, 'batch': 8, 'computed_kv_gb': 0.21875, 'measured_kv_gb': 0.2291259765625, 'peak_total_gb': 3.171963691711426} 3s elapsed
{'context': 2048, 'batch': 8, 'computed_kv_gb': 0.4375, 'measured_kv_gb': 0.4468994140625, 'peak_total_gb': 3.439663887023926} 5s elapsed
{'context': 4096, 'batch': 8, 'computed_kv_gb': 0.875, 'measured_kv_gb': 0.8829345703125, 'peak_total_gb': 3.975552558898926} 9s elapsed
{'context': 8192, 'batch': 8, 'computed_kv_gb': 1.75, 'measured_kv_gb': 1.7579345703125, 'peak_total_gb': 5.043423652648926} 22s elapsed
{'context': 16384, 'batch': 8, 'computed_kv_gb': 3.5, 'measured_kv_gb': 3.5079345703125, 'peak_total_gb': 7.184536933898926} 70s elapsed
{'context': 32768, 'batch': 8, 'compu

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>